Starting code

First, we import the packages we will need. Add any other packages here if need be. 

In [ ]:
import pandas as pd
import re 

We load our data into a dataframe object

In [ ]:
# Example dataframe
df = pd.read_csv('qualtrics.csv')  # Replace with your actual data source

We write our mapping between times and session id

In [ ]:
# --- Your predetermined sessions --- need to replace this with your actual session data
# Set your starting time of the session later than it's real starting time and the ending time a bit later. 
# The idea is just that the starting and ending times are the windows when people are actually submitting the survey.

sessions = pd.DataFrame({
    'session_id': [1, 2, 3, 4],
    'session_start': pd.to_datetime([
        '2024-01-15 09:00:00',
        '2024-01-15 13:00:00',
        '2024-01-16 10:00:00',
        '2024-01-17 15:00:00',
    ]),
    'session_end': pd.to_datetime([
        '2024-01-15 10:00:00',
        '2024-01-15 14:00:00',
        '2024-01-16 11:00:00',
        '2024-01-17 16:00:00',
    ])
})

In [ ]:

# Merge asof matches rows based on the nearest key instead of an exact match. However, 
# we also need to ensure that the submission falls within the session's end time.
# merge_asof requires both frames sorted by the join key
# You will have to adjust the column names to match your actual data. 
# For example, if your submission timestamp is in a column called 'timestamp', 
# you would replace 'submitted_at' with 'timestamp'.
df = df.sort_values('submitted_at').reset_index(drop=True) # Here the column name is assumed to be 'submitted_at', for example. 
sessions_sorted = sessions.sort_values('session_start').reset_index(drop=True)

# Match each submission to the most recent session_start at or before it
matched = pd.merge_asof(
    df,
    sessions_sorted,
    left_on='submitted_at',
    right_on='session_start',
    direction='backward'
)

# Now check that the timestamp actually falls within that session's end time too
# (otherwise a submission 3 days after a session would incorrectly match it)
matched['session_id'] = matched['session_id'].where(
    matched['submitted_at'] <= matched['session_end']
)

# This is our new dataframe with matched sessions, where session_id is NaN if no valid session was found
# For example, I think I submitted my own survey late because I saw the submit button had not worked, 
# so this code would discard it. By looking at how many sessions id's are NaN, 
# you can see how many submissions were not matched to a session.
print(matched[['participant_id', 'submitted_at', 'session_id']])


Now that we have all the participants matched to a session, we can start reordering the sessions so that they all match up. 

In [ ]:
# --- Example current data (slot/order-based columns) ---
# q1_1 means "question 1's answer about whichever excerpt was played 1st" Double check this otherwise the below logic breaks down

# --- Your order mapping ---
# session_id -> list of excerpt numbers in the order they were played
# e.g. session 1: excerpt 3 played first, excerpt 1 second, excerpt 2 third
excerpt_order = {
    1: [3, 1, 2],
    2: [1, 3, 2],
    # ... one entry per session, listing all 10 excerpts in play order
}

'''
If all this code looks very scary we can break it down into smaller pieces and explain each piece. 
The main idea is that we are taking a dataframe where the columns are based on the order of excerpts 
played in a session, and we want to transform it into a dataframe where the columns are based on the 
actual excerpt numbers, regardless of the order they were played in. I also was not able to run this on the 
real data, so there might be bugs. You can try to debug with AI, just be careful it knows excatly what 
you need it to do. Feel free to ask me questions otherwise. 
'''
# --- Step 1: identify all question/slot columns automatically ---
value_cols = [c for c in df.columns if re.match(r'^q\d+_\d+$', c)] # THIS MIGHT NEED TO BE ADJUSTED BASED ON YOUR ACTUAL COLUMN NAMES
id_cols = [c for c in df.columns if c not in value_cols]

# --- Step 2: melt to long format so each row is one participant/question/slot ---
long_df = df.melt(id_vars=id_cols, value_vars=value_cols,
                   var_name='q_col', value_name='response')

long_df[['question', 'slot']] = long_df['q_col'].str.extract(r'q(\d+)_(\d+)').astype(int)

# --- Step 3: map (session, slot) -> true excerpt number ---
def slot_to_excerpt(row):
    order = excerpt_order[row['session_id']]
    return order[row['slot'] - 1]  # slot is 1-indexed

long_df['excerpt'] = long_df.apply(slot_to_excerpt, axis=1)

# --- Step 4: build new excerpt-based column name, then pivot back to wide ---
long_df['q_col_new'] = 'q' + long_df['question'].astype(str) + '_excerpt' + long_df['excerpt'].astype(str)

wide_df = long_df.pivot_table(
    index=id_cols, columns='q_col_new', values='response', aggfunc='first'
).reset_index()

wide_df.columns.name = None

# Optional: sort columns nicely (id columns first, then q1_excerpt1...q1_excerpt10, q2_excerpt1...)
sorted_cols = id_cols + sorted(
    [c for c in wide_df.columns if c not in id_cols],
    key=lambda c: (int(re.search(r'q(\d+)_', c).group(1)), int(re.search(r'excerpt(\d+)', c).group(1)))
)
wide_df = wide_df[sorted_cols]

print(wide_df)
